## Basic Decoder-Only Transformer Implementation

In [1]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-21 07:28:53--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-03-21 07:28:53 (47.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [96]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [18]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [19]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [20]:
decode(encode('hello world'))

'hello world'

In [21]:
data = torch.tensor(encode(text), dtype=torch.long)

In [22]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [88]:
torch.manual_seed(100)
block_size = 8
batch_size = 4

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = data[torch.randint(0, len(data)-block_size, (batch_size, ))]

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])

  return x, y

In [94]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")

r -> t
t -> h
h -> e
e -> r
r -> ,
, ->  
  -> h
h -> e


In [104]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, vocab_size) # (vocab_size, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    logits = self.embedding(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # (B*T,)
      return logits, loss

    return logits, loss

  def generate(self, x, max_new_tokens):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      logits, _ = self(x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      l = logits[:, -1, :]


m = BigramLanguageModel()
logits, loss = m(xb, yb)
loss

tensor(4.9350, grad_fn=<NllLossBackward0>)